## Time travel — querying an old version of a table

Because the log is append-only, every past state of the table still exists — as long as the data files it referenced haven't been `VACUUM`-ed. You read a past state by **version number** or **timestamp**:

- **`VERSION AS OF n`** — the table as it was at commit `n`. Good for reproducibility: *what data did the model train on?*
- **`TIMESTAMP AS OF '2026-06-18 09:00:00'`** — the table as of a wall-clock time. Good for incident response: *what did the dashboard show before the bad backfill?*
- **`RESTORE TABLE ... TO VERSION AS OF n`** — writes a *new* commit that makes the current state equal to a past one. Time travel rewinds your read; `RESTORE` rewinds the table itself.

```sql
SELECT count(*) FROM silver.card_transactions VERSION AS OF 1;
SELECT * FROM silver.card_transactions TIMESTAMP AS OF '2026-06-18 09:00:00';
RESTORE TABLE silver.card_transactions TO VERSION AS OF 1;
```

Two table properties govern how far back you can go:

- `delta.logRetentionDuration` (default **30 days**) — how long commit JSONs are kept.
- `delta.deletedFileRetentionDuration` (default **7 days**) — how long `remove`-d files survive before `VACUUM` may delete them.

`VACUUM` aggressively and you lose the ability to travel further back than retention allows. Be deliberate.